# Naive Bayes Classifier — from scratch

Bayes' theorem gives the posterior of class $c$ given features $\mathbf{x}$:
$$P(c\mid\mathbf{x}) = \frac{P(\mathbf{x}\mid c)\,P(c)}{P(\mathbf{x})}$$

The **naive** assumption: features are conditionally independent given the class,
$P(\mathbf{x}\mid c) = \prod_j P(x_j\mid c)$. Predict the class with the largest posterior (working in log space to avoid underflow):

$$\hat{c} = \arg\max_c \Big[\log P(c) + \sum_j \log P(x_j\mid c)\Big]$$

Variants differ only in the likelihood $P(x_j \mid c)$:

| Variant | Feature type | Likelihood |
|---|---|---|
| **Gaussian NB** | continuous | $\mathcal{N}(x_j;\,\mu_{cj},\,\sigma^2_{cj})$ |
| **Multinomial NB** | counts (e.g. word counts) | $\theta_{cj} = \frac{N_{cj} + \alpha}{N_c + \alpha d}$ (Laplace smoothing) |
| **Categorical NB** | categorical | $\frac{\text{count}(x_j=v, c) + \alpha}{\text{count}(c) + \alpha\,|V_j|}$ |

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

import re

In [ ]:
def make_blobs(n_samples=300, centers=3, cluster_std=1.0, n_features=2,
               center_box=(-10, 10), random_state=None):
    """Isotropic Gaussian blobs (re-implementation of sklearn's make_blobs)."""
    rng = np.random.RandomState(random_state)
    if isinstance(centers, int):
        centers = rng.uniform(*center_box, size=(centers, n_features))
    centers = np.asarray(centers, dtype=float)
    k = len(centers)
    counts = [n_samples // k + (1 if i < n_samples % k else 0) for i in range(k)]
    stds = np.broadcast_to(np.asarray(cluster_std, dtype=float), (k,))
    X = np.vstack([rng.normal(c, s, size=(m, centers.shape[1]))
                   for c, s, m in zip(centers, stds, counts)])
    y = np.concatenate([np.full(m, i) for i, m in enumerate(counts)])
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def train_test_split(X, y, test_size=0.25, random_state=None):
    """Shuffle and split arrays into train and test subsets."""
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(X))
    n_test = int(round(len(X) * test_size))
    test, train = idx[:n_test], idx[n_test:]
    return X[train], X[test], y[train], y[test]

def accuracy_score(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))

def plot_decision_boundary(model, X, y, title="", ax=None, resolution=150):
    """Colour the plane by the model's prediction and overlay the data."""
    ax = ax or plt.gca()
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))
    Z = np.asarray(model.predict(np.c_[xx.ravel(), yy.ravel()]), dtype=float).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=25)
    ax.set_title(title)

## Gaussian Naive Bayes

In [ ]:
class GaussianNB:
    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.priors_ = np.array([np.mean(y == c) for c in self.classes_])
        self.means_ = np.array([X[y == c].mean(axis=0) for c in self.classes_])
        eps = self.var_smoothing * X.var(axis=0).max()
        self.vars_ = np.array([X[y == c].var(axis=0) for c in self.classes_]) + eps
        return self

    def _joint_log_likelihood(self, X):
        # log N(x; mu, var) summed over features, for every class -> (n_samples, n_classes)
        ll = -0.5 * (np.log(2 * np.pi * self.vars_)[None, :, :]
                     + (X[:, None, :] - self.means_[None, :, :]) ** 2 / self.vars_[None, :, :]).sum(axis=2)
        return ll + np.log(self.priors_)

    def predict_proba(self, X):
        jll = self._joint_log_likelihood(X)
        jll -= jll.max(axis=1, keepdims=True)          # log-sum-exp trick
        P = np.exp(jll)
        return P / P.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self._joint_log_likelihood(X), axis=1)]

In [ ]:
X, y = make_blobs(n_samples=600, centers=[[0, 0], [3, 3], [-3, 4]], cluster_std=[1.0, 1.5, 0.8], random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

gnb = GaussianNB().fit(X_train, y_train)
print("Class priors:", gnb.priors_.round(3))
print("Class means:\n", gnb.means_.round(3))
print("Test accuracy:", accuracy_score(y_test, gnb.predict(X_test)))
print("Probabilities for 3 test points:\n", gnb.predict_proba(X_test[:3]).round(4))

fig, ax = plt.subplots(figsize=(7, 5.5))
plot_decision_boundary(gnb, X_train, y_train, "Gaussian NB decision regions + class Gaussians", ax=ax)
t = np.linspace(0, 2 * np.pi, 100)
for mu, var in zip(gnb.means_, gnb.vars_):
    for k in [1, 2]:   # 1 and 2 standard-deviation ellipses (axis-aligned: features independent)
        ax.plot(mu[0] + k * np.sqrt(var[0]) * np.cos(t), mu[1] + k * np.sqrt(var[1]) * np.sin(t), "k", lw=0.8)
plt.show()

## Multinomial Naive Bayes for text: a tiny spam filter
We build the bag-of-words vectoriser from scratch too.

In [ ]:
train_texts = [
    "win money now", "free prize claim now", "cheap meds free offer", "win a free trip now",
    "limited offer win cash", "claim your free reward today", "urgent prize money waiting",
    "meeting at noon tomorrow", "project report attached", "lunch with the team today",
    "can we reschedule the meeting", "please review the attached report", "see you at lunch",
    "team meeting moved to friday",
]
train_labels = np.array(["spam"] * 7 + ["ham"] * 7)


def tokenize(text):
    return re.findall(r"[a-z']+", text.lower())


class CountVectorizer:
    def fit(self, docs):
        vocab = sorted({w for d in docs for w in tokenize(d)})
        self.vocab_ = {w: i for i, w in enumerate(vocab)}
        return self

    def transform(self, docs):
        M = np.zeros((len(docs), len(self.vocab_)))
        for r, d in enumerate(docs):
            for w in tokenize(d):
                if w in self.vocab_:               # unknown words are ignored
                    M[r, self.vocab_[w]] += 1
        return M


class MultinomialNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha                           # Laplace / additive smoothing

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.class_log_prior_ = np.log([np.mean(y == c) for c in self.classes_])
        counts = np.array([X[y == c].sum(axis=0) for c in self.classes_]) + self.alpha
        self.feature_log_prob_ = np.log(counts / counts.sum(axis=1, keepdims=True))
        return self

    def _jll(self, X):
        return X @ self.feature_log_prob_.T + self.class_log_prior_

    def predict_proba(self, X):
        jll = self._jll(X)
        P = np.exp(jll - jll.max(axis=1, keepdims=True))
        return P / P.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self._jll(X), axis=1)]

In [ ]:
vec = CountVectorizer().fit(train_texts)
X_bow = vec.transform(train_texts)
mnb = MultinomialNB(alpha=1.0).fit(X_bow, train_labels)
print("Vocabulary size:", len(vec.vocab_))

test_texts = ["free money offer", "report for the meeting", "win a prize today", "lunch tomorrow with team"]
probs = mnb.predict_proba(vec.transform(test_texts))
for text, pred, p in zip(test_texts, mnb.predict(vec.transform(test_texts)), probs):
    print(f"{text!r:32} -> {pred:4s}  P(ham)={p[0]:.3f}  P(spam)={p[1]:.3f}")

In [ ]:
# most indicative words for each class: log P(w|spam) - log P(w|ham)
inv_vocab = np.array(sorted(vec.vocab_, key=vec.vocab_.get))
ratio = mnb.feature_log_prob_[1] - mnb.feature_log_prob_[0]
order = np.argsort(ratio)
plt.figure(figsize=(8, 5))
idx = np.r_[order[:8], order[-8:]]
plt.barh(inv_vocab[idx], ratio[idx], color=["tab:blue"] * 8 + ["tab:red"] * 8)
plt.xlabel("log P(word|spam) - log P(word|ham)"); plt.title("Most ham-like (blue) and spam-like (red) words")
plt.tight_layout(); plt.show()

### Why smoothing?
Without smoothing ($\alpha=0$) a word never seen in a class gets probability 0, so $\log 0 = -\infty$ wipes out all other evidence.

In [ ]:
with np.errstate(divide="ignore"):
    raw = MultinomialNB(alpha=0).fit(X_bow, train_labels)

w = vec.vocab_["meeting"]
print("alpha=0: P('meeting' | spam) =", np.exp(raw.feature_log_prob_[1, w]))
print("alpha=1: P('meeting' | spam) =", np.exp(mnb.feature_log_prob_[1, w]).round(4))

doc = vec.transform(["free prize money at the meeting"])[0]
present = doc > 0                                   # only words that occur contribute
for name, model in [("alpha=0", raw), ("alpha=1", mnb)]:
    scores = model.class_log_prior_ + (doc[present] * model.feature_log_prob_[:, present]).sum(axis=1)
    print(f"{name}: log-scores {dict(zip(model.classes_.tolist(), scores.round(2).tolist()))}")
print("With alpha=0 both classes score -inf, so the classifier cannot decide.")

## Categorical Naive Bayes on Play Tennis

In [ ]:
class CategoricalNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.priors_ = {c: np.mean(y == c) for c in self.classes_}
        self.values_ = [np.unique(X[:, j]) for j in range(X.shape[1])]
        self.likelihood_ = {}                         # (feature, value, class) -> P(value | class)
        for c in self.classes_:
            Xc = X[y == c]
            for j, vals in enumerate(self.values_):
                for v in vals:
                    self.likelihood_[(j, v, c)] = (np.sum(Xc[:, j] == v) + self.alpha) / (len(Xc) + self.alpha * len(vals))
        return self

    def predict_log_scores(self, x):
        with np.errstate(divide="ignore"):          # alpha=0 can give log(0) = -inf, which is intended
            return {c: np.log(self.priors_[c]) + sum(np.log(self.likelihood_.get((j, v, c), 1e-9))
                                                     for j, v in enumerate(x))
                    for c in self.classes_}

    def predict(self, X):
        preds = []
        for x in X:
            scores = self.predict_log_scores(x)
            preds.append(max(scores, key=scores.get))
        return np.array(preds)


data = np.array([
    ["Sunny", "Hot", "High", "Weak", "No"], ["Sunny", "Hot", "High", "Strong", "No"],
    ["Overcast", "Hot", "High", "Weak", "Yes"], ["Rain", "Mild", "High", "Weak", "Yes"],
    ["Rain", "Cool", "Normal", "Weak", "Yes"], ["Rain", "Cool", "Normal", "Strong", "No"],
    ["Overcast", "Cool", "Normal", "Strong", "Yes"], ["Sunny", "Mild", "High", "Weak", "No"],
    ["Sunny", "Cool", "Normal", "Weak", "Yes"], ["Rain", "Mild", "Normal", "Weak", "Yes"],
    ["Sunny", "Mild", "Normal", "Strong", "Yes"], ["Overcast", "Mild", "High", "Strong", "Yes"],
    ["Overcast", "Hot", "Normal", "Weak", "Yes"], ["Rain", "Mild", "High", "Strong", "No"],
])
cnb = CategoricalNB(alpha=0).fit(data[:, :-1], data[:, -1])
query = ["Sunny", "Cool", "High", "Strong"]
scores = cnb.predict_log_scores(query)
print("Query:", query)
for c, s in scores.items():
    print(f"  P({c}) * prod P(x_j|{c}) = {np.exp(s):.5f}")
print("Prediction:", cnb.predict(np.array([query]))[0])
print("Training accuracy:", np.mean(cnb.predict(data[:, :-1]) == data[:, -1]))

## Summary
* Naive Bayes is fast (one pass over the data), works well with little data and with many features (text).
* The independence assumption is rarely true, yet the **ranking** of classes is often still correct.
* Probabilities can be over-confident; use log-space and smoothing for numerical robustness.